# Single Task Visualization by Corner (Process-Separated)

This notebook visualizes adaptation for each corner (FF, SS, TT) separately.

**Task Definition per Corner:**
- FF: 4 support → 2 test (same corner, different voltage/temp)
- SS: 4 support → 2 test (same corner, different voltage/temp)
- TT: 2 support → 1 test (same corner, different voltage)

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import numpy as np
import random
import re
import matplotlib.pyplot as plt
from pathlib import Path
from collections import OrderedDict, defaultdict

# Add paths
sys.path.insert(0, '/home/tkdgn2907/Deepsets_test/MAML/Projects/model_code')
from maml_optimized import OptimizedMAML, MAMLModel_3hidden

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

DATA_DIR = '/home/tkdgn2907/Deepsets_test/MAML/Projects/CAD_TEST/AND_cells_extracted'
DATA_TYPE = 'cell'
MODEL_PATH = '/home/tkdgn2907/Deepsets_test/MAML/Projects/pretrained_models/training_loss_taskdivide_all/cell_innerdiv100_meta32_combined_519traintask_full1DMAML_weights_3hidden_(40)_300000_inner1_upgraded_tsmc.pth'
TSMC_TRAIN_INPUT_PATH = '/home/tkdgn2907/Deepsets_test/MAML/Projects/dataset_all/MLP_dataset_TSMC/combined_data/tsmc_topology_agnostic_train_input_cell.pth'

GPU_ID = '0'
RANDOM_TASK_ID = None  # Set to specific number or None for random

# Corner splits
CORNER_SPLITS = {
    'FF': {
        'support': ['ff0p88vm40c', 'ff0p99v125c', 'ff1p1v125c', 'ff1p1vm40c'],
        'test': ['ff0p88v125c', 'ff0p99vm40c']
    },
    'SS': {
        'support': ['ss0p72vm40c', 'ss0p81v125c', 'ss0p9v125c', 'ss0p9vm40c'],
        'test': ['ss0p72v125c', 'ss0p81vm40c']
    },
    'TT': {
        'support': ['tt0p8v25c', 'tt0p9v25c'],
        'test': ['tt1p0v25c']
    }
}

# TSMC Process Parameters
PARAM_A = [1.427, 1.457, 1.430, 1.470, 1.443, 1.483, 1.43, 1.47, 1.43, 1.47]
PARAM_B = [0.026, 0.045, 0, 0, -0.026, -0.05, 0.0208, -0.04, 0.036, -0.0208]
PARAM_C = [0.024, 2.000, 0.024, 2.000, 0.024, 2.000, 0.024, 2.000, 0.024, 2.000]
CORNER_TO_IDX = {'FF': 0, 'TT': 1, 'SS': 2, 'FS': 3, 'SF': 4}

In [ ]:
# GPU settings
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Device: {device}')

In [ ]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def parse_filename(filename):
    match = re.search(r'(ff|tt|ss|fs|sf)(\d+p\d+)v(m?\d+)c', filename.lower())
    if not match:
        raise ValueError(f"Cannot parse filename: {filename}")
    corner = match.group(1).upper()
    voltage = float(match.group(2).replace('p', '.'))
    temp_str = match.group(3)
    temperature = -float(temp_str[1:]) if temp_str.startswith('m') else float(temp_str)
    return corner, voltage, temperature

def get_abc_params(corner):
    idx = CORNER_TO_IDX.get(corner.upper(), 1)
    nmos_idx, pmos_idx = idx * 2, idx * 2 + 1
    return {'a_n': PARAM_A[nmos_idx], 'a_p': PARAM_A[pmos_idx],
            'b_n': PARAM_B[nmos_idx], 'b_p': PARAM_B[pmos_idx],
            'c_n': PARAM_C[nmos_idx], 'c_p': PARAM_C[pmos_idx]}

def parse_lib_file(lib_path):
    with open(lib_path, 'r') as f:
        content = f.read()
    samples = []
    for cell_match in re.finditer(r'cell\s*\((\w+)\)\s*\{', content):
        cell_name = cell_match.group(1)
        cell_start = cell_match.end()
        brace_count = 1
        for i in range(cell_start, len(content)):
            if content[i] == '{': brace_count += 1
            elif content[i] == '}':
                brace_count -= 1
                if brace_count == 0:
                    cell_end = i
                    break
        cell_content = content[cell_start:cell_end]
        for timing_match in re.finditer(r'timing\s*\(\)\s*\{\s*related_pin\s*:\s*"(\w+)"', cell_content):
            related_pin = timing_match.group(1)
            timing_start = timing_match.end()
            t_brace = 1
            for i in range(timing_start, len(cell_content)):
                if cell_content[i] == '{': t_brace += 1
                elif cell_content[i] == '}':
                    t_brace -= 1
                    if t_brace == 0:
                        timing_end = i
                        break
            timing_content = cell_content[timing_start:timing_end]
            for delay_type in ['cell_rise', 'cell_fall']:
                table_match = re.search(rf'{delay_type}\s*\([^)]+\)\s*\{{', timing_content)
                if table_match:
                    table_start = table_match.end()
                    tb = 1
                    for i in range(table_start, len(timing_content)):
                        if timing_content[i] == '{': tb += 1
                        elif timing_content[i] == '}':
                            tb -= 1
                            if tb == 0:
                                table_end = i
                                break
                    table_content = timing_content[table_start:table_end]
                    idx1 = re.search(r'index_1\s*\(\s*"([^"]+)"\s*\)', table_content)
                    idx2 = re.search(r'index_2\s*\(\s*"([^"]+)"\s*\)', table_content)
                    vals = re.search(r'values\s*\(\s*(.*?)\s*\)\s*;', table_content, re.DOTALL)
                    if idx1 and idx2 and vals:
                        index_1 = [float(x.strip()) for x in idx1.group(1).split(',')]
                        index_2 = [float(x.strip()) for x in idx2.group(1).split(',')]
                        values_str = vals.group(1).replace('\\', '').replace('\n', ' ')
                        rows = re.findall(r'"([^"]+)"', values_str)
                        values = [[float(x.strip()) for x in row.split(',')] for row in rows]
                        samples.append({'cell_name': cell_name, 'delay_type': delay_type,
                                       'related_pin': related_pin, 'index_1': index_1,
                                       'index_2': index_2, 'values': values})
    return samples

def create_mlp_input(samples, corner, voltage, temperature):
    abc_params = get_abc_params(corner)
    inputs, outputs, metadata = [], [], []
    for sample in samples:
        if sample['delay_type'] not in ['cell_rise', 'cell_fall']:
            continue
        delay_indicator = -1 if 'rise' in sample['delay_type'] else 1
        a = (abc_params['a_n'] + abc_params['a_p']) / 2
        b = abc_params['b_n'] + abc_params['b_p']
        c = abc_params['c_n'] + abc_params['c_p']
        for ri, slew in enumerate(sample['index_1']):
            for ci, load in enumerate(sample['index_2']):
                if ri < len(sample['values']) and ci < len(sample['values'][ri]):
                    inputs.append([a, b, c, temperature, voltage, 2, delay_indicator, slew, load])
                    outputs.append([sample['values'][ri][ci]])
                    metadata.append({'cell_name': sample['cell_name'], 'delay_type': sample['delay_type'],
                                    'related_pin': sample['related_pin'], 'slew_idx': ri, 'load_idx': ci,
                                    'slew': slew, 'load': load})
    return (torch.tensor(inputs, dtype=torch.float32),
            torch.tensor(outputs, dtype=torch.float32), metadata) if inputs else (torch.tensor([]), torch.tensor([]), [])

In [ ]:
# ============================================================
# LOAD ALL DATA
# ============================================================

print("Loading data...")
base_path = Path(DATA_DIR)
all_data = {}

for voltage_dir in ['0p8v', '0p9v', '1p0v']:
    dir_path = base_path / voltage_dir
    if not dir_path.exists():
        continue
    for lib_file in sorted(dir_path.glob('AND_*.tlib')):
        match = re.search(r'AND_lib1_(\w+)_base_400\.tlib', lib_file.name)
        if not match:
            continue
        condition = match.group(1)
        corner, voltage, temperature = parse_filename(lib_file.name)
        samples = parse_lib_file(str(lib_file))
        inputs, outputs, metadata = create_mlp_input(samples, corner, voltage, temperature)
        if len(inputs) > 0:
            all_data[condition] = (inputs, outputs, metadata, corner, voltage, temperature)
            print(f"  {condition}: {len(inputs)} samples ({corner}, {voltage}V, {temperature}C)")

print(f"\nTotal conditions loaded: {len(all_data)}")

In [ ]:
# ============================================================
# APPLY TSMC NORMALIZATION
# ============================================================

print("Loading TSMC normalization stats...")
tsmc_train = torch.load(TSMC_TRAIN_INPUT_PATH)
norm_indices = [3, 4, 7, 8]
feature_names = {3: 'temperature', 4: 'voltage', 7: 'slew', 8: 'load'}
tsmc_norm_stats = {}

print("\nTSMC Normalization Stats:")
for idx in norm_indices:
    mean = tsmc_train[:, :, idx].mean().item()
    std = tsmc_train[:, :, idx].std().item()
    tsmc_norm_stats[idx] = (mean, std)
    print(f"  {feature_names[idx]}: mean={mean:.6f}, std={std:.6f}")
del tsmc_train

def apply_norm(data, stats):
    for idx, (mean, std) in stats.items():
        if std > 0:
            data[:, idx] = (data[:, idx] - mean) / std

print("\nApplying normalization...")
for cond in all_data:
    inputs, outputs, metadata, corner, voltage, temp = all_data[cond]
    apply_norm(inputs, tsmc_norm_stats)
print("Done!")

In [ ]:
# ============================================================
# BUILD TASK INDEX
# ============================================================

def build_index(data_dict):
    index = defaultdict(lambda: defaultdict(dict))
    for cond, (inputs, outputs, metadata, corner, voltage, temp) in data_dict.items():
        for idx, meta in enumerate(metadata):
            cell_key = (meta['cell_name'], meta['delay_type'], meta['related_pin'])
            sl_key = (meta['slew_idx'], meta['load_idx'])
            index[cell_key][cond][sl_key] = idx
    return index

data_index = build_index(all_data)
print(f"Total unique (cell, delay, pin) combinations: {len(data_index)}")

In [ ]:
# ============================================================
# LOAD MODEL
# ============================================================

print(f"Loading model: {MODEL_PATH}")
maml_model = OptimizedMAML(
    model=MAMLModel_3hidden(in_features=9, layer_length=40),
    dataset_in=None, dataset_out=None, inner_lr=0.001, meta_lr=0.0001
)
state_dict = torch.load(MODEL_PATH, map_location=device)
maml_model.model.load_state_dict(state_dict)
maml_model.model.to(device)
maml_model.model.eval()
print("Model loaded!")

In [ ]:
# ============================================================
# SELECT RANDOM TASK
# ============================================================

# Find a task that exists in all conditions
all_conditions = set()
for split in CORNER_SPLITS.values():
    all_conditions.update(split['support'])
    all_conditions.update(split['test'])

valid_tasks = []
for cell_key in data_index:
    sl_keys = None
    for cond in all_conditions:
        if cond in data_index[cell_key]:
            cond_sl = set(data_index[cell_key][cond].keys())
            sl_keys = cond_sl if sl_keys is None else sl_keys & cond_sl
    if sl_keys:
        for sl_key in sl_keys:
            if all(cond in data_index[cell_key] and sl_key in data_index[cell_key][cond] 
                   for cond in all_conditions):
                valid_tasks.append((cell_key, sl_key))

print(f"Total valid tasks across all corners: {len(valid_tasks)}")

if RANDOM_TASK_ID is None:
    task_idx = random.randint(0, len(valid_tasks) - 1)
else:
    task_idx = RANDOM_TASK_ID

cell_key, sl_key = valid_tasks[task_idx]

print("\n" + "=" * 80)
print(f"Selected Task: {task_idx} / {len(valid_tasks)}")
print("=" * 80)
print(f"  Cell: {cell_key[0]}")
print(f"  Delay: {cell_key[1]}")
print(f"  Pin: {cell_key[2]}")
print(f"  Slew idx: {sl_key[0]}, Load idx: {sl_key[1]}")

In [ ]:
# ============================================================
# DISPLAY ALL DATA FOR THIS TASK
# ============================================================

print("\nALL DATA FOR THIS TASK:")
print("=" * 100)
print(f"{'Condition':<15} {'Corner':<6} {'Voltage':<8} {'Temp':<8} {'Output (ns)':<12} {'Role'}")
print("-" * 100)

task_data = {}
for cond in sorted(all_conditions):
    inputs, outputs, metadata, corner, voltage, temp = all_data[cond]
    sample_idx = data_index[cell_key][cond][sl_key]
    output_val = outputs[sample_idx].item()
    
    # Determine role
    role = "?"
    for c, split in CORNER_SPLITS.items():
        if cond in split['support']:
            role = f"{c} Support"
        elif cond in split['test']:
            role = f"{c} Test"
    
    task_data[cond] = {'corner': corner, 'voltage': voltage, 'temp': temp, 'output': output_val, 'role': role}
    print(f"{cond:<15} {corner:<6} {voltage:<8.2f} {temp:<8.0f} {output_val:<12.6f} {role}")

In [ ]:
# ============================================================
# ADAPTATION FUNCTION
# ============================================================

def run_adaptation_with_tracking(initial_model, X_support, y_support, X_query, y_query,
                                  max_steps=100, lr=3e-3):
    model = nn.Sequential(OrderedDict([
        ('l1', nn.Linear(9, 40)), ('relu1', nn.ReLU()),
        ('l2', nn.Linear(40, 40)), ('relu3', nn.ReLU()),
        ('l4', nn.Linear(40, 40)), ('relu2', nn.ReLU()),
        ('l3', nn.Linear(40, 1))
    ])).to(device)
    model.load_state_dict(initial_model.state_dict())

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    y_mean, y_std = y_support.mean(), y_support.std()
    if y_std < 1e-8:
        y_std = torch.tensor(1.0)
    y_target = (y_support - y_mean) / y_std

    support_losses, query_losses, query_mapes = [], [], []
    predictions_over_time = []

    for step in range(max_steps + 1):
        with torch.no_grad():
            pred_support = model(X_support)
            support_loss = criterion(pred_support, y_target).item()
            support_losses.append(support_loss)

            pred_query = model(X_query) * y_std + y_mean
            query_mse = ((pred_query - y_query) ** 2).mean().item()
            query_losses.append(query_mse)

            mape = (torch.abs(pred_query - y_query) / (y_query + 1e-8)).mean().item() * 100
            query_mapes.append(mape)

            predictions_over_time.append(pred_query.cpu().numpy().flatten())

        if step < max_steps:
            model.zero_grad()
            loss = criterion(model(X_support), y_target)
            loss.backward()
            optimizer.step()

    return {
        'support_losses': support_losses,
        'query_losses': query_losses,
        'query_mapes': query_mapes,
        'predictions_over_time': predictions_over_time,
        'actual_values': y_query.cpu().numpy().flatten()
    }

In [ ]:
# ============================================================
# RUN ADAPTATION FOR EACH CORNER
# ============================================================

corner_results = {}

for corner_name, split in CORNER_SPLITS.items():
    print(f"\n{'='*60}")
    print(f"Processing {corner_name} Corner")
    print(f"  Support: {split['support']}")
    print(f"  Test: {split['test']}")
    print(f"{'='*60}")
    
    # Build support set
    X_support_list, y_support_list, support_info = [], [], []
    for cond in split['support']:
        inputs, outputs, metadata, corner, voltage, temp = all_data[cond]
        sample_idx = data_index[cell_key][cond][sl_key]
        X_support_list.append(inputs[sample_idx:sample_idx+1])
        y_support_list.append(outputs[sample_idx:sample_idx+1])
        support_info.append({'condition': cond, 'voltage': voltage, 'temp': temp, 
                            'output': outputs[sample_idx].item()})
    
    # Build query set
    X_query_list, y_query_list, query_info = [], [], []
    for cond in split['test']:
        inputs, outputs, metadata, corner, voltage, temp = all_data[cond]
        sample_idx = data_index[cell_key][cond][sl_key]
        X_query_list.append(inputs[sample_idx:sample_idx+1])
        y_query_list.append(outputs[sample_idx:sample_idx+1])
        query_info.append({'condition': cond, 'voltage': voltage, 'temp': temp,
                          'output': outputs[sample_idx].item()})
    
    X_support = torch.cat(X_support_list, dim=0).to(device)
    y_support = torch.cat(y_support_list, dim=0).to(device)
    X_query = torch.cat(X_query_list, dim=0).to(device)
    y_query = torch.cat(y_query_list, dim=0).to(device)
    
    print(f"\n  Support: {len(split['support'])} samples")
    for info in support_info:
        print(f"    {info['condition']}: {info['voltage']}V, {info['temp']}C -> {info['output']:.6f} ns")
    
    print(f"\n  Query: {len(split['test'])} samples")
    for info in query_info:
        print(f"    {info['condition']}: {info['voltage']}V, {info['temp']}C -> {info['output']:.6f} ns")
    
    # Run adaptation
    results = run_adaptation_with_tracking(
        maml_model.model.model, X_support, y_support, X_query, y_query, max_steps=100
    )
    
    corner_results[corner_name] = {
        'results': results,
        'support_info': support_info,
        'query_info': query_info
    }
    
    # Print final results
    final_mape = results['query_mapes'][40]
    print(f"\n  Result (step 40): MAPE = {final_mape:.2f}%")

In [ ]:
# ============================================================
# VISUALIZATION: MAPE OVER STEPS FOR EACH CORNER
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colors = {'FF': 'green', 'SS': 'red', 'TT': 'blue'}

for idx, (corner_name, data) in enumerate(corner_results.items()):
    ax = axes[idx]
    results = data['results']
    
    ax.plot(results['query_mapes'], color=colors[corner_name], linewidth=1.5)
    ax.axvline(x=40, color='gray', linestyle='--', alpha=0.5, label='Step 40')
    ax.axhline(y=results['query_mapes'][40], color='gray', linestyle=':', alpha=0.5)
    
    ax.set_xlabel('Adaptation Step')
    ax.set_ylabel('Query MAPE (%)')
    ax.set_title(f'{corner_name} Corner\n(Support: {len(data["support_info"])} → Test: {len(data["query_info"])})')
    ax.grid(True, alpha=0.3)
    
    # Add annotation
    ax.annotate(f'MAPE@40: {results["query_mapes"][40]:.2f}%', 
                xy=(40, results['query_mapes'][40]), xytext=(50, results['query_mapes'][40]+5),
                fontsize=10, color=colors[corner_name])

plt.suptitle(f'Task: {cell_key[0]} / {cell_key[1]} / slew={sl_key[0]}, load={sl_key[1]}', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# VISUALIZATION: PREDICTIONS VS ACTUAL FOR EACH CORNER
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (corner_name, data) in enumerate(corner_results.items()):
    ax = axes[idx]
    results = data['results']
    query_info = data['query_info']
    
    x_labels = [info['condition'] for info in query_info]
    x_pos = np.arange(len(x_labels))
    
    actuals = results['actual_values']
    preds_0 = results['predictions_over_time'][0]
    preds_40 = results['predictions_over_time'][40]
    
    width = 0.25
    ax.bar(x_pos - width, actuals, width, label='Actual', color='black', alpha=0.7)
    ax.bar(x_pos, preds_0, width, label='Step 0', color='lightgray', alpha=0.7)
    ax.bar(x_pos + width, preds_40, width, label='Step 40', color=colors[corner_name], alpha=0.7)
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Delay (ns)')
    ax.set_title(f'{corner_name} Corner')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Predictions vs Actual by Corner', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# DETAILED RESULTS TABLE
# ============================================================

print("\n" + "=" * 100)
print("DETAILED PREDICTION RESULTS (Step 40)")
print("=" * 100)

for corner_name, data in corner_results.items():
    print(f"\n{corner_name} Corner:")
    print("-" * 80)
    print(f"{'Condition':<15} {'Voltage':<8} {'Temp':<8} {'Actual':<12} {'Predicted':<12} {'Error (%)':<10}")
    print("-" * 80)
    
    results = data['results']
    query_info = data['query_info']
    preds = results['predictions_over_time'][40]
    actuals = results['actual_values']
    
    for i, info in enumerate(query_info):
        error_pct = abs(preds[i] - actuals[i]) / (actuals[i] + 1e-8) * 100
        print(f"{info['condition']:<15} {info['voltage']:<8.2f} {info['temp']:<8.0f} {actuals[i]:<12.6f} {preds[i]:<12.6f} {error_pct:<10.2f}%")
    
    mape = results['query_mapes'][40]
    print(f"\n  => MAPE: {mape:.2f}%")

In [ ]:
# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("SUMMARY: By Corner Results for This Task")
print("=" * 80)
print(f"\nTask: {cell_key[0]} / {cell_key[1]} / {cell_key[2]}")
print(f"Slew idx: {sl_key[0]}, Load idx: {sl_key[1]}")
print()
print(f"{'Corner':<10} {'Support':<10} {'Test':<10} {'MAPE@40 (%)':<15}")
print("-" * 45)

for corner_name, data in corner_results.items():
    n_support = len(data['support_info'])
    n_test = len(data['query_info'])
    mape = data['results']['query_mapes'][40]
    print(f"{corner_name:<10} {n_support:<10} {n_test:<10} {mape:<15.2f}")

In [ ]:
# ============================================================
# TT CORNER DETAILED ANALYSIS
# ============================================================

print("\n" + "=" * 80)
print("TT CORNER DETAILED ANALYSIS (2 Support → 1 Test)")
print("=" * 80)

tt_data = corner_results['TT']
tt_results = tt_data['results']

print("\nSupport Data:")
for info in tt_data['support_info']:
    print(f"  {info['condition']}: {info['voltage']}V, {info['temp']}C -> {info['output']:.6f} ns")

print("\nTest Data:")
for i, info in enumerate(tt_data['query_info']):
    actual = tt_results['actual_values'][i]
    pred = tt_results['predictions_over_time'][40][i]
    error = abs(pred - actual) / actual * 100
    print(f"  {info['condition']}: {info['voltage']}V, {info['temp']}C")
    print(f"    Actual: {actual:.6f} ns")
    print(f"    Predicted: {pred:.6f} ns")
    print(f"    Error: {error:.2f}%")

# Plot TT adaptation curve
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax1 = axes[0]
ax1.plot(tt_results['support_losses'], 'b-', label='Support Loss')
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')
ax1.set_title('TT: Support Loss')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)
ax1.axvline(x=40, color='r', linestyle='--', alpha=0.5)

ax2 = axes[1]
ax2.plot(tt_results['query_mapes'], 'g-', label='Query MAPE')
ax2.set_xlabel('Step')
ax2.set_ylabel('MAPE (%)')
ax2.set_title('TT: Query MAPE')
ax2.grid(True, alpha=0.3)
ax2.axvline(x=40, color='r', linestyle='--', alpha=0.5)
ax2.axhline(y=tt_results['query_mapes'][40], color='gray', linestyle=':', alpha=0.5)

plt.suptitle('TT Corner: 2 Support (0.8V, 0.9V) → 1 Test (1.0V)')
plt.tight_layout()
plt.show()